# 第四章 符号理解与生成：符号域音乐分析

本 Notebook 核对 `music21` 的和弦、调性与声部进行工具，并记录各自的适用域与错配情形。

## 结构

| 章节 | 内容 | 对应正文节 |
|:---:|:---|:---:|
| 0 | 环境自检 | — |
| 1 | 加载素材（贝多芬小步舞曲 / 巴赫 BWV 854 / 茉莉花） | — |
| 2 | 和弦与罗马数字功能标记 | 4.3.1 |
| 3 | 调性与音阶分析（Krumhansl-Schmuckler 调性剖面） | 4.3.2 |
| 4 | 复调与声部分离 | 4.3.3 |

**素材及用途**：
- **贝多芬 G 大调小步舞曲 WoO 10 No. 2** —— 分块和声织体，用于观察纵合切片与自定义按拍启发式的候选输出。
- **巴赫 BWV 854 赋格** —— 复调线性对位织体，用于检查 `chordify` 的适用边界。
- **《茉莉花》** —— 候选空间错配样本；Krumhansl-Schmuckler 调性剖面仅包含西方大小调候选。

## 第 0 部分 —— 环境自检

In [ ]:
import sys
from pathlib import Path

assert sys.version_info >= (3, 10), f"需要 Python >= 3.10, 当前为 {sys.version}"

import music21
import numpy as np
import matplotlib.pyplot as plt

print("python  :", sys.version.split()[0])
print("music21 :", music21.__version__)
print("numpy   :", np.__version__)

# 中文字体 / 白底配置
plt.rcParams['font.sans-serif'] = [
    'PingFang SC', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei',
    'Arial Unicode MS', 'Noto Sans CJK SC', 'DejaVu Sans',
]
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor']   = 'white'
plt.rcParams['axes.facecolor']     = 'white'
plt.rcParams['axes.edgecolor']     = 'black'
plt.rcParams['axes.labelcolor']    = 'black'
plt.rcParams['xtick.color']        = 'black'
plt.rcParams['ytick.color']        = 'black'
plt.rcParams['text.color']         = 'black'

# 项目根目录自动推断
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "CODE").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "CODE").is_dir(), "未找到 CODE/ 目录, 请在项目根目录启动 notebook"
print("项目根目录 :", PROJECT_ROOT)

FIG_DIR = PROJECT_ROOT / "CODE" / "chapter04" / "output_figures"
FIG_DIR.mkdir(exist_ok=True)
print("图像输出目录 :", FIG_DIR)

## 第 1 部分 —— 加载素材

贝多芬小步舞曲与《茉莉花》从 MIDI 读入；BWV 854 同时读入 MIDI 与配套 MusicXML。涉及拍号、拍位和原谱小节的分析使用 MusicXML，MIDI 只用于核对容器结构与冲突元数据。

MIDI 文件中的元数据（如轨道内的拍号、调号和文本类 meta event）需要与谱源逐项核对；标准 MIDI 文件头块本身只记录格式、轨道数与时间分辨率。本章两个本地文件都显示这种必要性：贝多芬文件的拍号元事件与出版谱冲突；BWV 854 的 SMF 元事件为 4/4，而配套 MusicXML 与谱例均为 3/4。BWV 854 的 MusicXML 第 1 小节从偏移 0 直接发声，SMF 首音前 2 秒只作为前导静音记录，不能解释为原谱空小节。本书用边车文件（sidecar）记录已有证据支持的校勘值；没有证据的来源与版本信息不补造。下一段单元内嵌这些校勘元数据。

In [ ]:
from music21 import converter, meter, key as m21key

# 人工校勘的边车文件（以 dict 形式内嵌，等价于一份独立的 YAML）
# 字段名保留英文作为内部数据结构键，值使用中文
BEETHOVEN_SIDECAR = {
    "work_id":                "WoO 10 No. 2",
    "composer":               "路德维希·凡·贝多芬",
    "title":                  "G 大调小步舞曲",
    "source_midi":            "CODE/datasets/lmd_clean_midi/Ludwig van Beethoven/"
                              "Beethoven Minuet in G major, WoO 10, No. 2.mid",
    "notated_meter":          "3/4",       # MIDI 拍号元事件错标为 4/4
    "notated_key":            "G major",   # MIDI 轨道中缺失调号元事件
    "analysis_tempo_bpm":     120,         # 文件无速度元事件时采用的播放回退值；不是出版谱速度证据
    "texture":                "带装饰音的分块和声织体",
    "metadata_corrections":   ["拍号", "调号"],
    "notes": "本地 MIDI 的拍号为 4/4; 两个独立出版谱版本均记为 3/4。"
             "调号元事件在 MIDI 轨道中缺失, 正确应为 G 大调(一个升号)。"
             "织体为古典主调织体, 含装饰音与经过音。",
}

BWV854_SIDECAR = {
    "work_id":                "BWV 854",
    "composer":               "约翰·塞巴斯蒂安·巴赫",
    "title":                  "E 大调前奏曲与赋格 —— 赋格",
    "source_midi":            "CODE/datasets/lmd_clean_midi/Bach Johann Sebastian/"
                              "Bach Prelude and Fugue in E major BWV 854 Fugue.mid",
    "source_musicxml":        "CODE/datasets/MusicXML/"
                              "Bach Prelude and Fugue in E major BWV 854 Fugue.musicxml",
    "notated_meter":          "3/4",
    "notated_key":            "E major",
    "texture":                "三声部线性对位",
    "notes": "三声部赋格; 本书配套 MusicXML 与谱例均为 3/4, 第 1 小节直接发声。"
             "本地 MIDI 的 4/4 拍号元事件与记谱冲突, 首音前 2 秒是 SMF 前导静音; MIDI 为单 Part 的多音叠加。",
}

JASMINE_SIDECAR = {
    "title":                  "茉莉花",
    "source_region":          "本地文件未记录；具体版本来源无法核实",
    "source_midi":            "CODE/datasets/melodies/茉莉花.midi",
    "notated_meter":          "2/4",
    "modal_analysis":         "本章分析口径：C 徵调式(F 宫系统；实测音级 F-G-A-C-D)",
    "western_profile_outputs": "本例三种大小调剖面输出 C 大调或 F 大调；均不等同于目标调式标签",
    "texture":                "单声部旋律",
    "notes": "本地片段只出现 F-G-A-C-D 五个音级；具体版本来源未记录。Krumhansl-Schmuckler 调性剖面预设了 12 个大调加 12 个小调, "
             "其候选空间中没有'徵调式'这一类别, 因此算法只能强制归入某个西方调。",
}

# 加载分析素材；source_key 明确区分 MIDI 与 MusicXML
def load_with_sidecar(sidecar, source_key="source_midi"):
    path = PROJECT_ROOT / sidecar[source_key]
    score = converter.parse(str(path))
    return score, path

beethoven, beethoven_path = load_with_sidecar(BEETHOVEN_SIDECAR)
bwv854_midi, bwv854_midi_path = load_with_sidecar(BWV854_SIDECAR, "source_midi")
bwv854, bwv854_score_path = load_with_sidecar(BWV854_SIDECAR, "source_musicxml")
jasmine, jasmine_path     = load_with_sidecar(JASMINE_SIDECAR)

print(f"贝多芬小步舞曲 : {beethoven_path.name}  音符数={len(list(beethoven.recurse().notes))}")
print(f"BWV 854 MIDI   : {bwv854_midi_path.name}    音符/和弦对象数={len(list(bwv854_midi.recurse().notes))}")
print(f"BWV 854 乐谱   : {bwv854_score_path.name}  音符/和弦对象数={len(list(bwv854.recurse().notes))}")
print(f"茉莉花         : {jasmine_path.name}   音符数={len(list(jasmine.recurse().notes))}")

### 1.1 应用边车文件中的校勘值

贝多芬文件的 4/4 拍号元事件与出版谱的 3/4 不一致；只替换 `TimeSignature` 不会重新划分已有小节，因此下面排除 MIDI 文件自身的前导空白，并按 3/4 重建小节。BWV 854 不对原始 MIDI 强行重切小节：该 SMF 的 4/4 元事件与 3/4 记谱冲突，而且 MIDI 时间单位与记谱四分音符并非可直接逐拍替换。本章凡涉及拍号、拍位和原谱小节的 BWV 854 分析均直接使用配套 MusicXML；原始 MIDI 只用于观察容器结构。

In [ ]:
import copy
from music21 import stream

def apply_sidecar_corrections(score, sidecar):
    """按边车证据校勘拍号/调号；拍号变化时重新划分小节。"""
    first_part = score.parts[0] if score.parts else score
    target_meter = sidecar.get("notated_meter")
    existing_ts = list(first_part.recurse().getElementsByClass(meter.TimeSignature))
    current_meter = existing_ts[0].ratioString if existing_ts else None

    if target_meter and current_meter != target_meter:
        # flatten() 保留层级换算后的绝对 offset。复制所有发声事件，
        # 排除本地文件开头的空白，再按目标拍号重切小节。
        events = list(first_part.flatten().notes)
        first_sound = min(float(event.offset) for event in events)
        rebuilt_flat = stream.Part(id=first_part.id or 'corrected_part')
        rebuilt_flat.insert(0.0, meter.TimeSignature(target_meter))
        if sidecar.get("notated_key"):
            tonic, mode = sidecar["notated_key"].split()
            rebuilt_flat.insert(0.0, m21key.Key(tonic, mode.lower()))
        for event in events:
            rebuilt_flat.insert(float(event.offset) - first_sound, copy.deepcopy(event))
        rebuilt_part = rebuilt_flat.makeMeasures(inPlace=False)
        rebuilt_part.makeTies(inPlace=True)
        rebuilt_score = stream.Score()
        rebuilt_score.insert(0.0, rebuilt_part)
        return rebuilt_score

    # 拍号已与证据一致：保留原有小节，只校勘调号元数据。
    for cls in (m21key.KeySignature, m21key.Key):
        for elem in list(first_part.recurse().getElementsByClass(cls)):
            first_part.remove(elem, recurse=True)
    if sidecar.get("notated_key"):
        tonic, mode = sidecar["notated_key"].split()
        first_part.insert(0.0, m21key.Key(tonic, mode.lower()))
    return score

beethoven = apply_sidecar_corrections(beethoven, BEETHOVEN_SIDECAR)
bwv854    = apply_sidecar_corrections(bwv854,    BWV854_SIDECAR)
# 茉莉花 MIDI 轨道已含 2/4 拍号与 F major 调号元事件, 不需校勘; 边车中的 modal_analysis 记录本章分析口径

# 打印校勘后的拍号/调号元数据和实际小节容量
for label, s in [("贝多芬", beethoven), ("BWV 854", bwv854), ("茉莉花", jasmine)]:
    ts = list(s.recurse().getElementsByClass(meter.TimeSignature))
    ks = list(s.recurse().getElementsByClass((m21key.KeySignature, m21key.Key)))
    measures = list(s.parts[0].getElementsByClass(stream.Measure)) if s.parts else []
    first_lengths = [float(m.barDuration.quarterLength) for m in measures[:3]]
    print(f"{label:10s} 拍号={ts[0].ratioString if ts else '-':>4s}   "
          f"调号={ks[0] if ks else '-'}   前三小节容量={first_lengths}")

---

## 第 2 部分 —— 和弦与罗马数字功能标记

**纵合切片**（vertical slice）：在每个时间位置汇集所有同时发声的音。`music21.stream.Score.chordify()` 将多声部谱面聚合为单一和弦事件流。

**罗马数字功能标记**（Roman numeral）：把绝对和弦（如 G-B-D）转换为调内的功能记号（如 I 级）。`music21.roman.romanNumeralFromChord(chord, key)` 在给定调性的前提下完成这一标注。

本节先在**贝多芬小步舞曲**（分块和声织体）上运行标准流程，再把相同方法用于 **BWV 854 赋格**（复调线性对位），检查工具链的适用域边界。

### 2.1 贝多芬小步舞曲：标准流程

流程分为三步：
1. `chordify()` 把整段谱面纵合为一串和弦（Chord 对象）。
2. `analyze('key')` 给出作品主调（用作罗马数字标注的参考调性）。
3. 对每个和弦调用 `romanNumeralFromChord(c, key)` 得到调内的功能记号。

In [ ]:
from music21 import roman, chord

def chordify_and_label(score, max_chords=None):
    """把 chordify 与 romanNumeralFromChord 一体化调用.

    返回 (主调, [(时间偏移, 和弦, 罗马数字记号), ...])
    """
    tonal_key = score.analyze('key')
    chordified = score.chordify()

    results = []
    for c in chordified.recurse().getElementsByClass(chord.Chord):
        if not c.pitches:
            continue
        try:
            rn = roman.romanNumeralFromChord(c, tonal_key)
            figure = rn.figure
        except Exception:
            figure = "?"
        results.append((float(c.offset), c, figure))
        if max_chords is not None and len(results) >= max_chords:
            break
    return tonal_key, results

beethoven_key, beethoven_labels = chordify_and_label(beethoven, max_chords=16)

print(f"贝多芬小步舞曲  识别主调 = {beethoven_key}")
print(f"前 {len(beethoven_labels)} 个和弦(时间偏移, 音高, 罗马数字记号):\n")
print(f"{'偏移':>6s}  {'音高':<22s}  {'罗马数字'}")
print(f"{'------':>6s}  {'-'*22}  {'------'}")
for offset, c, fig in beethoven_labels:
    pitches_str = "-".join(p.nameWithOctave for p in c.pitches)
    print(f"{offset:6.2f}  {pitches_str:<22s}  {fig}")

原始 chordify 把每一个纵合切片都当作独立和弦，因此装饰音、经过音等和弦外音都会被切成临时和弦（如上面时间偏移 2.25 处的 `G2-B♭4-B4-C#5-D5` → `i5#4b33`）。

下一段使用本章自定义的按拍启发式：先把跨越拍边界的纵合切片按实际重叠时值分摊到各拍，再在每拍内累计音级权重，并保留实际最低音与若干高权重音级。

In [ ]:
from collections import defaultdict
from music21 import pitch as m21pitch

def salience_weighted_beat_chords(score, top_k_pitch_classes=3, max_bars=None):
    """本例自定义的按拍时值加权启发式；返回 (参考调, 结果列表)。

    chordify 切片若跨越拍边界，先按它与各拍的实际重叠时值分摊权重。
    每个音级在相应拍内累加该重叠时值。重建代表和弦时
    保留该拍实际最低音，以免在只保留音级后伪造转位；其余位置选择高权重
    音级在该拍出现过的最高实际音。max_bars 表示最先发声的若干小节；
    输出小节号按发声顺序从 1 重排，因而不会把 MIDI 开头的空白小节算入。
    """
    tonal_key  = score.analyze('key')
    chordified = score.chordify()

    # pc_salience[(小节号, 拍号)] = {音级: 累积时值}
    pc_salience = defaultdict(lambda: defaultdict(float))
    observed = defaultdict(lambda: defaultdict(list))
    bass_by_beat = {}
    for c in chordified.recurse().getElementsByClass(chord.Chord):
        if not c.pitches:
            continue
        measure = c.measureNumber or 0
        start = float(c.offset)
        end = start + float(c.duration.quarterLength)
        beat_length = float(c.beatDuration.quarterLength)
        if beat_length <= 0 or end <= start:
            continue
        lowest = min(c.pitches, key=lambda p: p.ps)
        first_beat = int(np.floor(start / beat_length))
        last_beat = int(np.ceil(end / beat_length))
        for beat_zero_based in range(first_beat, last_beat):
            beat_start = beat_zero_based * beat_length
            beat_end = beat_start + beat_length
            overlap = max(0.0, min(end, beat_end) - max(start, beat_start))
            if overlap <= 1e-12:
                continue
            beat_key = (measure, beat_zero_based + 1)
            if beat_key not in bass_by_beat or lowest.ps < bass_by_beat[beat_key].ps:
                bass_by_beat[beat_key] = m21pitch.Pitch(lowest.nameWithOctave)
            for p in c.pitches:
                pc_salience[beat_key][p.pitchClass] += overlap
                observed[beat_key][p.pitchClass].append(m21pitch.Pitch(p.nameWithOctave))

    sounding_measures = sorted({measure for measure, _ in pc_salience})
    allowed_measures = set(sounding_measures[:max_bars]) if max_bars is not None else None
    display_measure = {measure: idx + 1 for idx, measure in enumerate(sounding_measures)}
    results = []
    for (measure, beat_idx), pc_weights in sorted(pc_salience.items()):
        if allowed_measures is not None and measure not in allowed_measures:
            continue
        bass = bass_by_beat[(measure, beat_idx)]
        ranked_pcs = [pc for pc, _ in sorted(pc_weights.items(), key=lambda kv: -kv[1])]
        selected_pcs = [bass.pitchClass] + [
            pc for pc in ranked_pcs if pc != bass.pitchClass
        ][:max(0, top_k_pitch_classes - 1)]
        if not selected_pcs:
            continue
        pitches = [bass] + [
            max(observed[(measure, beat_idx)][pc], key=lambda p: p.ps)
            for pc in selected_pcs[1:]
        ]
        representative = chord.Chord(pitches)
        try:
            rn = roman.romanNumeralFromChord(representative, tonal_key)
            figure = rn.figure
        except Exception:
            figure = "?"
        results.append((display_measure[measure], beat_idx, representative, figure))
    return tonal_key, results

beethoven_bar_key, beethoven_bar_labels = salience_weighted_beat_chords(
    beethoven, top_k_pitch_classes=3, max_bars=8)

print(f"贝多芬小步舞曲  调 = {beethoven_bar_key}")
print(f"前 8 小节按拍取显著度加权后的罗马数字:\n")
print(f"{'小节':>4s} {'拍':>4s}  {'代表音':<20s}  {'罗马数字'}")
print(f"{'-'*4} {'-'*4}  {'-'*12}  ------")
for measure, beat_idx, c, fig in beethoven_bar_labels:
    pitches_str = "-".join(p.nameWithOctave for p in c.pitches)
    print(f"{measure:>4d} {beat_idx:>4d}  {pitches_str:<12s}  {fig}")

下面的罗马数字是 `chordify`、本例按拍启发式和 `romanNumeralFromChord` 串联后的输出，不是人工标注真值。自然音或变音标签只能说明代表音集合在给定 G 大调参照下被如何解释。

**可视化：贝多芬前 8 小节的罗马数字序列**

深色柱表示按本例记号规则归为自然音和弦的候选，浅色柱表示带变音、离调指向或其他半音化记号的候选。

**图题：贝多芬 G 大调小步舞曲 WoO 10 No. 2 前 8 小节按拍罗马数字标注。**

In [ ]:
def plot_roman_numeral_sequence(labels, figsize=(11, 3.2),
                                highlight_diatonic=True):
    """把 (小节号, 拍号, 和弦, 罗马数字) 序列可视化为横向时间轴。

    自然音和弦用深色, 变音和弦(离调/调式交替/半音化)用浅色，
    以展示 music21 chordify 工具链在不同织体上的"生效率"。
    """
    fig, ax = plt.subplots(figsize=figsize)
    xs = list(range(len(labels)))

    # "自然音和弦 vs 变音和弦"判别(基于 music21 罗马数字)：
    #   自然音和弦(diatonic chord) = 根音为调内音(罗马数字首字符无 b/# 前缀)
    #                                且不出现任何变音记号(+/#/b)
    #   变音和弦(non-diatonic / altered chord) = 其余一切情形，涵盖
    #       · 离调(secondary)和弦(如 V/V)
    #       · 调式交替/借用和弦(mode-mixture，如大调中的 bIII、bVI)
    #       · 增六和弦(augmented sixth)
    #       · 纯半音化临时和弦(chordify 在复调织体上拟合出的非功能集合)
    #   判别采用罗马数字记号而非音乐学推理，因此是一个保守近似，
    #   不自动鉴定是否为真实离调；该判断需要结合上下文与人工和声分析。
    def _is_diatonic(fig_str):
        if fig_str.startswith(('b', '#')):                       # 根音已变音
            return False
        if '/' in fig_str:                                      # 副属/离调指向
            return False
        if any(ch in fig_str for ch in ('+', '#', 'b')):         # 含音程变音记号
            return False
        for p in ('IV', 'vii', 'iii', 'ii', 'vi', 'V', 'I'):     # 长 prefix 优先
            if fig_str.startswith(p):
                return True
        return False

    is_diatonic = [_is_diatonic(fig_str) for _, _, _, fig_str in labels]

    colors = ['#2b5f99' if (is_diatonic[i] and highlight_diatonic) else '#bcbcbc'
              for i in xs]
    ax.bar(xs, [1]*len(xs), color=colors, edgecolor='white', width=0.9)

    # 每条柱子上方贴罗马数字，下方贴小节和拍。标签较多时旋转罗马数字，
    # 避免长标签互相覆盖。
    dense_labels = len(labels) > 24
    for i, (measure, beat_idx, _, fig_str) in enumerate(labels):
        ax.text(i, 1.04, fig_str,
                ha='left' if dense_labels else 'center', va='bottom',
                rotation=65 if dense_labels else 0,
                fontsize=7 if dense_labels else 9,
                fontweight='bold' if is_diatonic[i] else 'normal')
        ax.text(i, -0.05, f"第{measure}小节第{beat_idx}拍",
                ha='right', va='top', rotation=45,
                fontsize=9, color='#555')

    # 仅为柱顶罗马数字和柱底小节标签预留空间，不再为整图标题留白。
    ax.set_ylim(-0.82, 1.45 if dense_labels else 1.20)
    ax.set_xlim(-0.5, len(xs) - 0.5)
    ax.set_yticks([])
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.tight_layout()
    return fig

fig = plot_roman_numeral_sequence(beethoven_bar_labels)

out_path = FIG_DIR / "fig_431_beethoven_roman.png"
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print("图像已保存 ->", out_path)

### 2.2 BWV 854 赋格：复调织体上的错配

把同一套自定义启发式搬到 BWV 854 上。`chordify` 只完成共时音集合的纵向聚合。赋格中的经过音、延留音和声部换音会产生大量短切片。下方输出用于展示这一具体片段和参数设置的边界。

柱形深浅的含义与上一图相同。

**图题：同一套自定义按拍启发式在 BWV 854 赋格第 1--8 小节上的罗马数字候选（按 3/4 记谱切分）。**

In [ ]:
bwv854_key, bwv854_labels = salience_weighted_beat_chords(
    bwv854, top_k_pitch_classes=3, max_bars=8)

print(f"BWV 854 赋格  识别主调 = {bwv854_key}")
print(f"前 {min(24, len(bwv854_labels))} 条按拍罗马数字:\n")
print(f"{'小节':>4s} {'拍':>4s}  {'代表音':<20s}  {'罗马数字'}")
print(f"{'-'*4} {'-'*4}  {'-'*12}  ------")
for measure, beat_idx, c, fig_str in bwv854_labels[:24]:
    pitches_str = "-".join(p.nameWithOctave for p in c.pitches)
    print(f"{measure:>4d} {beat_idx:>4d}  {pitches_str:<12s}  {fig_str}")

In [ ]:
fig = plot_roman_numeral_sequence(bwv854_labels, figsize=(12, 3.2))
out_path = FIG_DIR / "fig_431_bwv854_chordify_failure.png"
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print("图像已保存 ->", out_path)

---

## 第 3 部分 —— 调性与音阶分析

**Krumhansl-Schmuckler 调性剖面**：该方法源于 Krumhansl 与 Kessler 的探测音研究；`music21` 实现公开一条大调和一条小调的 12 维权重剖面，并循环移位形成 24 个候选。输入乐曲的音级统计向量 $\mathbf{h} \in \mathbb{R}^{12}$ 与这些候选逐一求皮尔逊相关系数，取相关系数最大的候选作为输出。

### 3.1 两段作品的调性剖面估计结果

music21 的 `score.analyze('key.krumhanslschmuckler')` 一行调用同时完成以下三步：
- 按 `quarterLength` 对音高类加权；Chord 中每个音都加上完整和弦时值
- 对 24 个 Krumhansl-Schmuckler 剖面循环求相关系数
- 返回最优候选调及相关系数（`correlationCoefficient` 属性）

In [ ]:
def analyze_with_multiple_profiles(score, methods=('key.krumhanslschmuckler',
                                                   'key.aardenessen',
                                                   'key.simple')):
    """用多个剖面方法对同一段乐曲估计调性, 便于并列对比."""
    rows = []
    for method in methods:
        try:
            k = score.analyze(method)
            rows.append((method, f"{k.tonic.name} {k.mode}",
                         getattr(k, 'correlationCoefficient', None)))
        except Exception as e:
            rows.append((method, f"错误: {e}", None))
    return rows

print("贝多芬小步舞曲:")
for method, key_label, cc in analyze_with_multiple_profiles(beethoven):
    cc_str = f"{cc:.3f}" if cc is not None else "-"
    print(f"  {method:30s} -> {key_label:10s}  相关系数={cc_str}")

print("\n《茉莉花》:")
for method, key_label, cc in analyze_with_multiple_profiles(jasmine):
    cc_str = f"{cc:.3f}" if cc is not None else "-"
    print(f"  {method:30s} -> {key_label:10s}  相关系数={cc_str}")

### 3.2 所有 24 个候选调的相关系数全景

`analyze('key')` 只返回最优候选。查看整条相关系数曲线可以比较各候选的相对接近程度，但该算法没有通用的绝对相关系数阈值来判定一段音乐是否属于候选空间。下一段按 music21 的时值加权口径自行计算 24 个候选并画图。

**图题：贝多芬小步舞曲与本地《茉莉花》文件在 24 个 Krumhansl-Schmuckler 候选调上的皮尔逊相关系数。**

In [ ]:
# Krumhansl-Kessler 调性剖面(1982 年心理实验拟合)
KS_MAJOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                     2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
KS_MINOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                     2.54, 4.75, 3.98, 2.69, 3.34, 3.17])

PC_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def pitch_class_histogram(score):
    """对 score 所有音符按持续时长加权, 输出归一化的 12 维音级直方图."""
    h = np.zeros(12, dtype=float)
    for n in score.recurse().notes:
        dur = float(n.duration.quarterLength)
        if n.isChord:
            for p in n.pitches:
                h[p.pitchClass] += dur
        else:
            h[n.pitch.pitchClass] += dur
    total = h.sum()
    return h / total if total > 0 else h

def ks_correlations(pc_hist):
    """对 24 个 Krumhansl-Schmuckler 剖面(12 大调 + 12 小调)求皮尔逊相关系数.

    返回 shape=(24,) 的数组: 索引 0-11 对应 C 大调 / C# 大调 / ... / B 大调;
    索引 12-23 对应 C 小调 / C# 小调 / ... / B 小调.
    """
    corrs = np.zeros(24)
    for tonic in range(12):
        # 剖面循环移位: 以 tonic 为 0 轴的 Krumhansl-Schmuckler 向量
        major_profile = np.roll(KS_MAJOR, tonic)
        minor_profile = np.roll(KS_MINOR, tonic)
        corrs[tonic]      = np.corrcoef(pc_hist, major_profile)[0, 1]
        corrs[tonic + 12] = np.corrcoef(pc_hist, minor_profile)[0, 1]
    return corrs

# 计算两段乐曲
beethoven_hist  = pitch_class_histogram(beethoven)
jasmine_hist    = pitch_class_histogram(jasmine)
beethoven_corrs = ks_correlations(beethoven_hist)
jasmine_corrs   = ks_correlations(jasmine_hist)

print("贝多芬小步舞曲  最优 3 个候选:")
idx_sorted = np.argsort(-beethoven_corrs)[:3]
for r, i in enumerate(idx_sorted, 1):
    mode  = "大调" if i < 12 else "小调"
    tonic = PC_NAMES[i % 12]
    print(f"  第 {r} 位  {tonic:>2s} {mode}  相关系数={beethoven_corrs[i]:+.3f}")

print("\n《茉莉花》  最优 3 个候选:")
idx_sorted = np.argsort(-jasmine_corrs)[:3]
for r, i in enumerate(idx_sorted, 1):
    mode  = "大调" if i < 12 else "小调"
    tonic = PC_NAMES[i % 12]
    print(f"  第 {r} 位  {tonic:>2s} {mode}  相关系数={jasmine_corrs[i]:+.3f}")

In [ ]:
def plot_ks_correlation_panorama(corrs_pair, labels):
    """把两段乐曲的 24 个 Krumhansl-Schmuckler 相关系数并排画在一张图上."""
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True)
    x = np.arange(24)
    xticklabels = [f"{PC_NAMES[i]} 大调" for i in range(12)] + \
                  [f"{PC_NAMES[i]} 小调" for i in range(12)]

    for ax, corrs, label in zip(axes, corrs_pair, labels):
        # 配色: 最优候选深蓝, 其余灰色; 负相关红色
        top_idx = int(np.argmax(corrs))
        colors = []
        for i, v in enumerate(corrs):
            if i == top_idx:
                colors.append('#1f4e79')
            elif v < 0:
                colors.append('#d97b7b')
            else:
                colors.append('#bcbcbc')
        ax.bar(x, corrs, color=colors, edgecolor='white', width=0.85)
        ax.axhline(0, color='#333', linewidth=0.5)
        ax.axvline(11.5, color='#888', linewidth=0.5, linestyle='--')
        ax.set_ylim(-0.6, 1.0)
        ax.set_ylabel('皮尔逊相关系数')
        ax.set_title(label, loc='left', fontsize=10)
        ax.text(5.5, 0.86, '大调候选', ha='center', fontsize=9, color='#555')
        ax.text(17.5, 0.86, '小调候选', ha='center', fontsize=9, color='#555')

    axes[-1].set_xticks(x)
    axes[-1].set_xticklabels(xticklabels, rotation=60, fontsize=9)
    plt.tight_layout()
    return fig

fig = plot_ks_correlation_panorama(
    [beethoven_corrs, jasmine_corrs],
    ["贝多芬 G 大调小步舞曲 WoO 10 No. 2",
     "《茉莉花》"])
out_path = FIG_DIR / "fig_432_ks_profile_jasmine.png"
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print("图像已保存 ->", out_path)

# 量化: 最优与次优的优势差距
for label, corrs in [("贝多芬", beethoven_corrs), ("茉莉花", jasmine_corrs)]:
    sorted_c = np.sort(corrs)[::-1]
    margin = sorted_c[0] - sorted_c[1]
    print(f"{label:6s}  最优={sorted_c[0]:+.3f}  次优={sorted_c[1]:+.3f}  "
          f"优势差距={margin:+.3f}")

---

## 第 4 部分 —— 复调与声部分离

### 4.1 MIDI 中的 Voice 与赋格声部不是同一回事

先看 BWV 854 MIDI 被 music21 解析之后的 Voice 对象：它在依据错误 4/4 元事件构造的**局部分组**内包含若干 Voice；多数这种分组恰好有 2--3 个，但这一数量不能证明它们就是作品的三条连续音乐学声部，也不能把这些分组称为原谱小节。Voice 的 `id` 是**局部的**——跨分组没有连续性保证，不同分组中的 voice1 可能对应不同的赋格声部。

这些 Voice 是 `music21` 在 SMF 派生分组内构造的记谱分层对象，数量不能直接解释为全曲独立声部数。标准 MIDI 事件本身没有“赋格第一/第二/第三声部”这样的全局语义标签；解析得到的 Voice `id` 也不能未经跨分组验证就当作连续声部身份。

In [ ]:
from music21 import stream

# 观察 1: 原始 part 中的 Voice 对象总数
voices_raw = list(bwv854_midi.parts[0].recurse().getElementsByClass(stream.Voice))

# 观察 2: makeVoices 之后的 Voice 对象总数
p_voiced    = bwv854_midi.parts[0].makeVoices(inPlace=False, fillGaps=True)
voices_made = list(p_voiced.recurse().getElementsByClass(stream.Voice))

# 观察 3: 各小节内部 Voice 数的分布(MIDI 解析已带)
from collections import Counter
voices_per_measure = Counter()
for m in bwv854_midi.parts[0].getElementsByClass('Measure'):
    voices_per_measure[len(list(m.getElementsByClass(stream.Voice)))] += 1

print(f"原始 part 中 Voice 对象总数:    {len(voices_raw)}")
print(f"makeVoices 后 Voice 对象总数: {len(voices_made)}")
print(f"(对象计数是否相等本身不能证明结构完全相同；需继续检查内容与跨小节连续性。)")
print(f"\nSMF 错误 4/4 元事件派生分组内部的 Voice 数分布:")
for n_voices, cnt in sorted(voices_per_measure.items()):
    print(f"  含 {n_voices} 个 Voice 的分组: {cnt} 个")

### 4.2 按音高分层的跨小节启发式分离

要得到**整曲级的三声部轨道**，本例在 MusicXML 的 3/4 记谱时间轴上按音高从高到低排序，再分派到上声部、中声部和下声部。实现还需处理三个边界，否则图中可能出现被误读为声部内音程的伪影。

**实现边界 1：按 offset 分组**。`score.flatten()` 会把层级 offset 换算为绝对 offset，并不会抹掉时间位置；问题在于逐个遍历 `flatten().notes` 时，同一 offset 的多个 Note 仍是多个独立事件。若算法需要共时切片，必须显式按 offset 分组，或像这里一样先用 `chordify()` 聚合。

**实现边界 2：2 音切片的声部分配**。本例明确约定把高音放上层、低音放下层、中层留空。它不是普遍的音乐学真值：三声部作品的两音时刻也可能是上+中或中+下。用 `round()` 做秩插值还会受到 Python ties-to-even 舍入规则影响，因此应把分配约定显式写出并单独测试。

**实现边界 3：`chordify` 的时间碎片化**。只要任一声部换音，`chordify` 就会切出新的 Chord。延音因而可能被切成多段，在三个子图中表现为一串紧邻的短条，容易被误读为声部内音程。分配完成后需做**延音合并**（tie merge），把同声部内首尾相接的同音高事件合并为长音。

**启发式的固有局限**：单音切片全部放到上层会系统性漏掉中、下声部的独奏时刻；多于目标声部数的音还会被丢弃。遇到声部交叉时，按瞬时音高排序也会交换声部身份。Chew--Wu contig mapping 把音符组织为 contig，并按连接代价的全局最小策略接续相邻 contig 中的音流；`partitura.musicanalysis.voice_separation.estimate_voices` 提供了该方法的可安装实现，其当前源码仍把倚音（grace note）处理列为需要改进的边界。

In [ ]:
from music21 import note as m21note

def pitch_height_voice_split(score, n_voices=3):
    """按音高层级把 score 拆成 n_voices 条独立 part(跨小节的整曲级).

    算法要点:
    - 音源: score.chordify().flatten() 得到带绝对偏移的 Chord 流.
      (直接逐个遍历 score.flatten().notes 未按相同 offset 分组, 详见上一段.)
    - 分配规则(保持音乐学合理性):
        1 音切片: 只填上声部(本例分配规则，会漏分其他声部的独奏时刻)
        2 音切片: 上声部 + 下声部，中声部留空(本例分配规则)
        3+ 音切片: 上声部 + 下声部, 中间音按音高秩线性插值至中声部
    - 碎片合并: chordify 会在任何声部换音时切新 Chord, 导致一个延音被
      切成多段. 最后做延音合并: 同声部内若相邻事件音高相同且紧邻,
      则合并为一段长音.
    """
    parts = [stream.Part(id=f'voice_{i}') for i in range(n_voices)]

    for c in score.chordify().flatten().getElementsByClass(chord.Chord):
        if not c.pitches:
            continue
        offset  = float(c.offset)
        dur     = float(c.duration.quarterLength)
        pitches = sorted(c.pitches, key=lambda x: x.ps, reverse=True)  # 高 -> 低
        n_p     = len(pitches)

        assignment = {}  # 声部索引 -> 音高
        if n_p == 1:
            assignment[0] = pitches[0]
        elif n_p == 2:
            assignment[0]            = pitches[0]    # 上声部 = 最高音
            assignment[n_voices - 1] = pitches[-1]   # 下声部 = 最低音
            # 中声部留空
        else:
            assignment[0]            = pitches[0]
            assignment[n_voices - 1] = pitches[-1]
            middle = pitches[1:-1]                   # 去掉最高最低后剩下的
            mid_slots = n_voices - 2
            for k, v_idx in enumerate(range(1, n_voices - 1)):
                idx = (int(k * (len(middle) - 1) / max(mid_slots - 1, 1))
                       if mid_slots > 1 else len(middle) // 2)
                if middle:
                    assignment[v_idx] = middle[min(idx, len(middle) - 1)]

        for v_idx, p in assignment.items():
            parts[v_idx].insert(offset, m21note.Note(p, quarterLength=dur))

    # 延音合并: 把相邻的同音高事件合并(去除 chordify 碎片化)
    for v_idx, part in enumerate(parts):
        notes = sorted([(float(n.offset), n.pitch.ps,
                         float(n.duration.quarterLength))
                        for n in part.recurse().notes])
        merged = stream.Part(id=part.id)
        i = 0
        while i < len(notes):
            off, ps, dur = notes[i]
            j = i + 1
            while (j < len(notes) and notes[j][1] == ps
                   and abs(notes[j][0] - (off + dur)) < 1e-3):
                dur += notes[j][2]
                j += 1
            merged.insert(off, m21note.Note(ps, quarterLength=dur))
            i = j
        parts[v_idx] = merged

    return parts

bwv854_voices = pitch_height_voice_split(bwv854, n_voices=3)

for i, p in enumerate(bwv854_voices):
    notes = list(p.recurse().notes)
    if notes:
        ps      = [n.pitch.ps for n in notes]
        offsets = [float(n.offset) for n in notes]
        durs    = [float(n.duration.quarterLength) for n in notes]
        print(f"声部 {i} ({p.id:10s}): 事件数={len(notes):>3d}  "
              f"MIDI 音高 {min(ps):>3.0f}-{max(ps):>3.0f} (均值 {np.mean(ps):5.1f})  "
              f"偏移 {min(offsets):>5.1f}-{max(offsets):>5.1f}  "
              f"平均时值={np.mean(durs):.2f}")

**可视化：三声部钢琴卷帘**。三个子图分别显示上声部、中声部和下声部。

**图题：BWV 854 赋格第 1--12 小节的音高分层钢琴卷帘（按 3/4 记谱切分）。**

In [ ]:
def plot_voices_piano_roll(voice_parts, titles, x_start, x_end,
                            measure_length=3.0, pad_y=3, figsize=(11, 6)):
    """分别绘制三个声部的钢琴卷帘. x 轴限定在 [x_start, x_end] 四分音符.

    y 轴根据窗口内的实际音高范围自适应, 上下各加 pad_y 个半音.
    """
    # 第一遍: 收集窗口内的实际音高, 决定 y 轴范围
    pitches_all = []
    for part in voice_parts:
        for n in part.recurse().notes:
            if x_start <= float(n.offset) < x_end:
                pitches_all.append(n.pitch.ps)
    y_lo = min(pitches_all) - pad_y
    y_hi = max(pitches_all) + pad_y

    fig, axes = plt.subplots(len(voice_parts), 1, figsize=figsize,
                             sharex=True, sharey=True)
    if len(voice_parts) == 1:
        axes = [axes]

    for ax, part, title in zip(axes, voice_parts, titles):
        for n in part.recurse().notes:
            x0  = float(n.offset)
            dur = float(n.duration.quarterLength)
            if x0 + dur <= x_start or x0 >= x_end:
                continue
            # 裁剪到窗口
            x_left  = max(x0, x_start)
            x_right = min(x0 + dur, x_end)
            ax.barh(n.pitch.ps, x_right - x_left, left=x_left,
                    height=0.85, color='#333333', edgecolor='none')
        ax.set_ylim(y_lo, y_hi)
        ax.set_xlim(x_start, x_end)
        ax.set_ylabel('MIDI 音高')
        ax.set_title(title, loc='left', fontsize=10)
        ax.grid(True, axis='y', linestyle=':', alpha=0.3)
        for boundary in np.arange(x_start, x_end + 1e-9, measure_length):
            ax.axvline(boundary, color='#999999', linewidth=0.6,
                       linestyle='--', alpha=0.55, zorder=0)

    axes[-1].set_xticks(np.arange(x_start, x_end + 1e-9, measure_length))
    axes[-1].set_xlabel(
        f'MusicXML 时间偏移（以四分音符为单位，{x_start:.0f}-{x_end:.0f}；每 3 个单位为一个 3/4 小节）')
    plt.tight_layout()
    return fig

# BWV 854 的 MusicXML 为 3/4，第 1 小节从偏移 0 直接发声。
# 选取原谱第 1--12 小节：3/4 × 12 = 36 quarterLength，即 [0, 36]。
fig = plot_voices_piano_roll(
    bwv854_voices,
    titles=['上声部（音高最高）',
            '中声部',
            '下声部（音高最低）'],
    x_start=0.0, x_end=36.0)
out_path = FIG_DIR / "fig_433_bwv854_voices.png"
fig.savefig(out_path, dpi=600, bbox_inches='tight')
plt.show()
print("图像已保存 ->", out_path)

### 4.3 声部进行检查：`VoiceLeadingQuartet`

声部分离是第一步。得到独立声部后，就可以用 `music21.voiceLeading.VoiceLeadingQuartet` 检查**两声部之间**是否出现传统对位与和声写作规范所关注的平行五度、平行八度等关系。

VoiceLeadingQuartet 接受四个音（前两个是声部 1 的前后两音，后两个是声部 2 的前后两音），然后查询它们是否构成 `parallelFifth()` / `parallelOctave()` / `hiddenFifth()` / `hiddenOctave()` 等关系。

In [ ]:
from music21 import voiceLeading

# 合成例：构造一个明确的平行五度对，核对 API 返回值
# VoiceLeadingQuartet 签名: (v1n1, v1n2, v2n1, v2n2)
# 即(上声部前/后音, 下声部前/后音)
# 上声部: G4 -> A4; 下声部: C4 -> D4; 每一步相距纯五度(7 个半音) => 平行五度
v1_n1 = m21note.Note('G4')
v1_n2 = m21note.Note('A4')
v2_n1 = m21note.Note('C4')
v2_n2 = m21note.Note('D4')

quartet = voiceLeading.VoiceLeadingQuartet(v1_n1, v1_n2, v2_n1, v2_n2)
print("--- 合成例: 上声部 G4->A4 / 下声部 C4->D4 ---")
print(f"  parallelFifth()   = {quartet.parallelFifth()}")
print(f"  parallelOctave()  = {quartet.parallelOctave()}")
print(f"  similarMotion()   = {quartet.similarMotion()}")
print(f"  (预期: 平行五度=True, 平行八度=False)")

# 真实作品: 在 BWV 854 的上声部 + 下声部中扫描相邻事件
def find_parallel_issues(voice_a, voice_b, min_offset=0, max_offset=36):
    """对两条声部成对的相邻事件构造 VoiceLeadingQuartet, 汇总平行五度与平行八度.

    voice_a 为上声部(对应 VoiceLeadingQuartet 中的 v1),
    voice_b 为下声部(对应 v2).
    """
    notes_a = sorted([n for n in voice_a.recurse().notes
                      if min_offset <= float(n.offset) < max_offset],
                     key=lambda n: n.offset)
    notes_b = sorted([n for n in voice_b.recurse().notes
                      if min_offset <= float(n.offset) < max_offset],
                     key=lambda n: n.offset)

    issues = []
    for i in range(len(notes_a) - 1):
        a1, a2 = notes_a[i], notes_a[i + 1]
        # 在 voice_b 中找到偏移最接近 a1 / a2 的音
        b1 = min(notes_b, key=lambda n: abs(n.offset - a1.offset))
        b2 = min(notes_b, key=lambda n: abs(n.offset - a2.offset))
        if b1 is b2:
            continue
        q = voiceLeading.VoiceLeadingQuartet(a1, a2, b1, b2)
        if q.parallelFifth() or q.parallelOctave():
            issues.append({
                'offset': (a1.offset, a2.offset),
                'upper':  (a1.nameWithOctave, a2.nameWithOctave),
                'lower':  (b1.nameWithOctave, b2.nameWithOctave),
                'parallel_fifth':  q.parallelFifth(),
                'parallel_octave': q.parallelOctave(),
            })
    return issues

top_voice = bwv854_voices[0]      # 上声部
bot_voice = bwv854_voices[-1]     # 下声部
issues = find_parallel_issues(top_voice, bot_voice, min_offset=0, max_offset=36)

print(f"\n--- BWV 854 第 1--12 个 3/4 小节（36 个四分音符时值）：上/下层启发式结果检查 ---")
print(f"检测到 {len(issues)} 处候选平行五度或平行八度；须与人工声部标注核对:")
for it in issues[:8]:
    kind = "平行五度" if it['parallel_fifth'] else "平行八度"
    print(f"  偏移 {it['offset'][0]:5.2f}->{it['offset'][1]:5.2f}   "
          f"上声部 {it['upper'][0]:>4s}->{it['upper'][1]:<4s}  "
          f"下声部 {it['lower'][0]:>4s}->{it['lower'][1]:<4s}  "
          f"{kind}")

**局限与后续工作**：上面只得到启发式分层后的候选位置；没有人工连续声部标注，无法判断其中哪些是真实平行、哪些是分离误差。可靠的“声部分离 → 声部进行检查”管线需要先验证声部身份。

**谱系引用**：
- **Chew-Wu contig mapping**（Chew & Wu 2004）：把音符组织为 contig，再按连接代价的全局最小策略接续相邻 contig 中的音流。
- **McLeod-Steedman HMM**（McLeod & Steedman 2016）：用概率模型刻画声部延续性。